## 데이터 로딩 + 확인

* csv 파일 로딩
* 데이터 확인은 다음의 함수로 한다.
    * sample(또는 head), info, describe

In [12]:
# 데이터 로딩

import pandas as pd

df = pd.read_csv('Chemical_Numeric_Data_Quality.csv', encoding='cp949')


In [13]:
# 가독성 향상을 위해 소수점 출력 포맷을 소수점 넷째자리까지 나오도록 설정
pd.set_option('display.float_format', lambda x: '%.4f' % x)

In [14]:
# 데이터 확인
df.sample(3)
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Lot            1000 non-null   int64  
 1   Temp(°C)       870 non-null    float64
 2   Viscosity(cP)  911 non-null    float64
 3   Failure        1000 non-null   object 
 4   Failure code   161 non-null    float64
dtypes: float64(3), int64(1), object(1)
memory usage: 39.2+ KB


,Lot,Temp(°C),Viscosity(cP),Failure code
count,1000.0000,870.0000,911.0000,161.0000
mean,2110175932.5680,31.4714,5.5346,3.1056
std,677478.9190,33.0484,11.8892,1.3991
min,2109021818.0000,2.4500,0.7400,1.0000
25%,2109585209.0000,24.3525,1.5700,2.0000
50%,2110189868.0000,25.4300,2.3900,3.0000
75%,2110782208.0000,26.4400,3.2500,4.0000
max,2111289797.0000,435.3500,101.8000,5.0000


## 데이터 정리
* 컬럼 이름에서 데이터의 단위 표기를 제거하기 위해 컬럼 이름을 변경한다.
* 실습 데이터의 FailureCode 데이터가 교재에 나온 데이터 형태와 달라서 그 형태를 맞춘다.

In [19]:
# 컬럼 이름 변경
df.columns = ['Lot', 'Temp', 'Viscosity', 'Failure', 'FailureCode']

# 변경 확인
df.head()

,Lot,Temp,Viscosity,Failure,FailureCode
0,2110262249,24.4400,101.8000,1,2.0000
1,2109975633,23.9500,82.6300,1,NaN
2,2109024936,23.2400,1.5200,1,4.0000
3,2110009104,NaN,81.4900,1,NaN
4,2109169908,23.3300,71.9200,1,NaN


In [20]:
# 실습용 데이터가 교재 실습 데이터와 형태가 달라 맞춰주는 코드
# 지금 이해하지 않아도 괜찮겠다.
df.FailureCode = df.FailureCode.fillna('None').astype(str)
df.FailureCode = df.FailureCode.apply(lambda x: x if x=='None' else str(int(float(x))))

In [24]:
# 데이터 확인 -> 작업 결과 확인
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          1000 non-null   int64  
 1   Temp         870 non-null    float64
 2   Viscosity    911 non-null    float64
 3   Failure      1000 non-null   object 
 4   FailureCode  1000 non-null   object 
dtypes: float64(2), int64(1), object(2)
memory usage: 39.2+ KB


,Lot,Temp,Viscosity,Failure,FailureCode
0,2110262249,24.4400,101.8000,1,2
1,2109975633,23.9500,82.6300,1,None
2,2109024936,23.2400,1.5200,1,4
3,2110009104,NaN,81.4900,1,None
4,2109169908,23.3300,71.9200,1,None


## STEP ① 완전성 지표

In [25]:
# 결측치 확인
df.isnull().sum()

# 위 코드는 아래 코드를 단계별로 실행한 것과 같다.
# df.isnull() : df 전체를 대상으로 각 칸이 결측치이면 True,
#               값이 있으면 False 반환
# sum() : isnull()의 결과 dataframe의 각 열 데이터 개수를 센다.
#       : False는 0, True는 1로 계산된다.
#       : 결과적으로는 각 열의 결측치 개수가 된다.

,0
Lot,0
Temp,130
Viscosity,89
Failure,0
FailureCode,0


In [27]:
# 결측치 처리
# df에 데이터 행에 하나라도 결측치가 있으면 행 전체를 삭제
df = df.dropna(how='any', axis=0)

# 결과 확인
df.isnull().sum()

,0
Lot,0
Temp,0
Viscosity,0
Failure,0
FailureCode,0


## STEP ② 유효성 지표

In [28]:
# 이상치
# 온도 20 미만 또는 온도 30 초과
# 또는 점도 3 초과이면 제거 대상으로 선택한다.
# 결과를 outlier에 저장한다.
# 이 줄에서는 아직 df의 행을 삭제하지 않았다.
outlier = df[(df.Temp < 20) | (df.Temp > 30) | (df.Viscosity > 3)]

# 이 코드를 이해하려면 Boolean Indexing을 알아야 한다.
# - df[조건]은 조건이 True인 행만 선택한다.
# - 조건이 여러개 일 때,
#       : 각 비교식은 괄호로 감싼다.
#       : |는 '또는(OR)'으로 조건들을 연결한다.

# 결과 확인
print(outlier)

            Lot    Temp  Viscosity Failure FailureCode
0    2110262249 24.4400   101.8000       1           2
1    2109975633 23.9500    82.6300       1        None
4    2109169908 23.3300    71.9200       1        None
6    2109032490 54.1100     2.1000      정상        None
7    2109332619 24.3900    71.1400       1        None
..          ...     ...        ...     ...         ...
938  2111185019 26.3600     3.0100      정상        None
947  2111211907 23.6100     3.5500      정상        None
951  2111214194 27.1000    41.0500      불량        None
961  2111224103 24.0500    42.4500       1        None
980  2111256812 44.2200    51.6800      불량        None

[296 rows x 5 columns]


In [29]:
# df에서 outlier에 해당하는 행을 삭제
df = df.drop(outlier.index, axis=0)

In [30]:
# 지금까지 작업한 후 데이터의 정보
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 495 entries, 2 to 986
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          495 non-null    int64  
 1   Temp         495 non-null    float64
 2   Viscosity    495 non-null    float64
 3   Failure      495 non-null    object 
 4   FailureCode  495 non-null    object 
dtypes: float64(2), int64(1), object(2)
memory usage: 23.2+ KB


## STEP ③ 일관성 지표

In [31]:
#
df.Failure.value_counts()

,count
Failure,
0,412
정상,45
1,33
불량,5


In [ ]:
#
df.Failure = df.Failure.map({'정상': 0, '0': 0, '불량': 1, '1': 1})

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 495 entries, 2 to 986
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          495 non-null    int64  
 1   Temp         495 non-null    float64
 2   Viscosity    495 non-null    float64
 3   Failure      495 non-null    int64  
 4   FailureCode  495 non-null    object 
dtypes: float64(2), int64(2), object(1)
memory usage: 23.2+ KB


In [ ]:
# 결과 확인
df.Failure.value_counts()

,count
Failure,
0,457
1,38


### STEP ④ 유일성 지표

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 495 entries, 2 to 986
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          495 non-null    int64  
 1   Temp         495 non-null    float64
 2   Viscosity    495 non-null    float64
 3   Failure      495 non-null    int64  
 4   FailureCode  495 non-null    object 
dtypes: float64(2), int64(2), object(1)
memory usage: 23.2+ KB


In [ ]:
# 중복값이 있는지 확인
df.Lot.duplicated().sum()

np.int64(40)

In [ ]:
# 중복값 삭제
df = df.drop_duplicates(subset=['Lot'], keep='first')

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 455 entries, 2 to 975
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          455 non-null    int64  
 1   Temp         455 non-null    float64
 2   Viscosity    455 non-null    float64
 3   Failure      455 non-null    int64  
 4   FailureCode  455 non-null    object 
dtypes: float64(2), int64(2), object(1)
memory usage: 21.3+ KB


### STEP ⑤ 정확성 지표

In [ ]:
# 데이터 확인
df.info()
df.sample(5)

,Lot,Temp,Viscosity,Failure,FailureCode
692,2110649000,26.7900,1.5400,0,None
423,2109458048,24.3500,2.4900,0,None
863,2110581504,27.0800,0.8700,0,None
320,2110580603,25.7000,2.8800,0,None
724,2111228330,25.8000,1.4100,0,None


In [ ]:
# 데이터에서 현장을 반영하지 못하는 데이터를 삭제한다.
# CASE1
incorrect_condi1_df = df[(df.Failure == 0) & (df.FailureCode != 'None')]
df.loc[incorrect_condi1_df.index, 'FailureCode'] = 'None'

# CASE2
incorrect_condi2_df = df[(df.Failure == 1) & (df.FailureCode == 'None')]
df.drop(incorrect_condi2_df.index, axis=0, inplace = True)

/tmp/ipykernel_3152/1453254836.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop(incorrect_condi2_df.index, axis=0, inplace = True)


In [ ]:
# 결과 확인
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 449 entries, 2 to 975
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   Lot          449 non-null    int64  
 1   Temp         449 non-null    float64
 2   Viscosity    449 non-null    float64
 3   Failure      449 non-null    int64  
 4   FailureCode  449 non-null    object 
dtypes: float64(2), int64(2), object(1)
memory usage: 21.0+ KB
